<a href="https://colab.research.google.com/github/dhanusharer/DL-practise/blob/main/keras05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***sequence to sequence***

In [ ]:
from keras.models import Model
from keras.layers import Input ,Dense,LSTM

In [ ]:
import pandas as pd
batch_size=64
epochs=100
latent_dim=256
num_samples=10000
data_path='english_french.csv'

In [ ]:
import pandas as pd # Ensure pandas is available for reading the CSV

input_data=[]
target_texts=[] # Renamed to `target_texts` to store all target sequences
input_character=set()
target_character=set()
seen_inputs = set() # To track unique input sentences to avoid duplicates

# Use pandas to read the CSV, which handles quoted fields with internal commas correctly.
# Added on_bad_lines='skip' to handle parsing errors by skipping problematic lines.
df = pd.read_csv(data_path, on_bad_lines='skip')

# Iterate over the specified number of samples from the DataFrame
for i in range(len(df)):
    input_text_current = df.iloc[i, 0] # Get input text (English) from the first column
    target_text_part_current = df.iloc[i, 1] # Get target text part (French) from the second column

    # Only process unique input sentences up to num_samples
    if input_text_current not in seen_inputs and len(input_data) < num_samples:
        seen_inputs.add(input_text_current)

        # Re-assemble target_text with a tab at the beginning and a newline at the end
        # Ensure target_text_part_current is a string before concatenation
        formatted_target_text_current = '\t' + str(target_text_part_current) + '\n'

        input_data.append(input_text_current)
        target_texts.append(formatted_target_text_current) # Append to the list of target texts

        for char in input_text_current:
            input_character.add(char)
        for char in formatted_target_text_current:
            target_character.add(char)
    elif len(input_data) >= num_samples:
        break # Stop if we have enough unique samples


In [ ]:
# Save the training model
model.save('s2s_training_model.keras')

# Save the encoder and decoder inference models
encoder_model.save('s2s_encoder_model.keras')
decoder_model.save('s2s_decoder_model.keras')

print('Models saved successfully!')

NameError: name 'model' is not defined

In [ ]:
import json

# Save character mappings
with open('input_token_index.json', 'w') as f:
    json.dump(input_token_index, f)

with open('target_token_index.json', 'w') as f:
    json.dump(target_token_index, f)

with open('reverse_input_char_index.json', 'w') as f:
    json.dump(reverse_input_char_index, f)

with open('reverse_target_char_index.json', 'w') as f:
    json.dump(reverse_target_char_index, f)

# Save model parameters
model_params = {
    'latent_dim': latent_dim,
    'num_encoder_tokens': num_encoder_tokens,
    'num_decoder_tokens': num_decoder_tokens,
    'max_encoder_seq_length': max_encoder_seq_length,
    'max_decoder_seq_length': max_decoder_seq_length
}
with open('s2s_model_params.json', 'w') as f:
    json.dump(model_params, f)

print('Character mappings and model parameters saved successfully!')

To load the models and parameters in a new session:

```python
from keras.models import load_model, Model
from keras.layers import Input, Dense, LSTM
import json

# Load model parameters
with open('s2s_model_params.json', 'r') as f:
    model_params = json.load(f)
latent_dim = model_params['latent_dim']
num_encoder_tokens = model_params['num_encoder_tokens']
num_decoder_tokens = model_params['num_decoder_tokens']
max_encoder_seq_length = model_params['max_encoder_seq_length']
max_decoder_seq_length = model_params['max_decoder_seq_length']

# Load character mappings
with open('input_token_index.json', 'r') as f:
    input_token_index = json.load(f)
with open('target_token_index.json', 'r') as f:
    target_token_index = json.load(f)
with open('reverse_input_char_index.json', 'r') as f:
    reverse_input_char_index = json.load(f)
with open('reverse_target_char_index.json', 'r') as f:
    reverse_target_char_index = json.load(f)

# Load the encoder model
encoder_model = load_model('s2s_encoder_model.keras')

# Load the decoder model
decoder_model = load_model('s2s_decoder_model.keras')

# You might also want to load the full training model if needed
# training_model = load_model('s2s_training_model.keras')

print('Models and parameters loaded successfully!')

# Re-define decode_sequence function (as it depends on the loaded models and parameters)
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    target_seq[0, 0, target_token_index['\t']] = 1.

    decoded_sentence = ''
    stop_condition = False
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        if (sampled_char == '\n' or len(decoded_sentence) > max_decoder_seq_length):
            stop_condition = True

        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.
        states_value = [h, c]
    return decoded_sentence
```

In [ ]:
input_character

In [ ]:
input_character=sorted(list(input_character))
target_charater=sorted(list(target_character))
num_encoder_tokens=len(input_character)
num_decoder_tokens=len(target_character)
max_encoder_seq_length=max([len(txt) for txt in input_data])
max_decoder_seq_length=max([len(txt) for txt in target_texts])


In [ ]:
len(input_data),num_encoder_tokens,num_decoder_tokens,max_encoder_seq_length,max_decoder_seq_length

In [ ]:
input_token_index=dict(
    (char,i)
    for i,char in enumerate(input_character)
)
target_token_index=dict(
    (char,i)
    for i,char in enumerate(target_character)
)

In [ ]:
import numpy as np

In [ ]:
encoder_input_data = np.zeros(
    (len(input_data), max_encoder_seq_length, num_encoder_tokens),
    dtype="float32"
)

decoder_input_data = np.zeros(
    (len(input_data), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32"
)

decoder_target_data = np.zeros(
    (len(input_data), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32"
)


In [ ]:
for i, (input_text, target_text) in enumerate(zip(input_data, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.
    encoder_input_data[i, t + 1:, input_token_index[' ']] = 1.

    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.
    decoder_input_data[i, t + 1:, target_token_index[' ']] = 1.
    decoder_target_data[i, t:, target_token_index[' ']] = 1.

In [ ]:
encoder_input_data[0].shape

In [ ]:
# Define an input sequence and process it.
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)

# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]


In [ ]:
# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = Input(shape=(None, num_decoder_tokens))

# We set up our decoder to return full output sequences,
# and to return internal states as well.
# We don't use the return states in the training model,
# but we will use them in inference.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs, initial_state=encoder_states
)

decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)


In [ ]:
# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Run training
model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder setup
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)


In [ ]:
reverse_input_char_index = dict(
    (i, char) for char, i in input_token_index.items()
)

reverse_target_char_index = dict(
    (i, char) for char, i in target_token_index.items()
)


In [ ]:
import numpy as np

def decode_sequence(input_seq):

    # Encode input sentence
    states_value = encoder_model.predict(input_seq)

    # Start with start token "\t"
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    target_seq[0, 0, target_token_index['\t']] = 1.

    decoded_sentence = ""
    stop_condition = False

    while not stop_condition:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        # Get predicted character
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]

        decoded_sentence += sampled_char

        # Stop if end token or too long
        if (sampled_char == '\n' or
           len(decoded_sentence) > max_decoder_seq_length):
            stop_condition = True

        # Update target_seq
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.

        # Update states
        states_value = [h, c]

    return decoded_sentence

In [ ]:
for seq_index in range(10):

    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)

    print("Input:", input_data[seq_index])
    print("Predicted:", decoded_sentence.strip())
    print("-" * 40)


In [ ]:
for seq_index in range(10):

    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)

    print("Input:", input_data[seq_index])
    print("Predicted:", decoded_sentence.strip())
    print("-" * 40)


# Task
Increase the `latent_dim` to 512 and `epochs` to 200, then retrain the sequence-to-sequence model and re-evaluate the translation output.

## Adjust Hyperparameters

### Subtask:
Increase the `latent_dim` (hidden units in LSTM) from 256 to 512, and increase the number of `epochs` from 100 to 200. This will give the model more capacity to learn and more time to train.


**Reasoning**:
The subtask requires increasing the `latent_dim` and `epochs` hyperparameters. I will update the code in cell `aiRbP3T9Vp31` to reflect these new values.



In [ ]:
import pandas as pd
batch_size=64
epochs=200
latent_dim=512
num_samples=10000
data_path='english_french.csv'

**Reasoning**:
Now that the hyperparameters `latent_dim` and `epochs` have been updated, I need to re-define the encoder model to reflect the new `latent_dim` value. This cell defines the encoder's input and LSTM layer.



In [ ]:
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)

# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]

**Reasoning**:
With the encoder model redefined, the decoder model also needs to be updated to incorporate the new `latent_dim`. This involves re-instantiating the `decoder_lstm` and `decoder_dense` layers with the correct `latent_dim` and connecting them to the `encoder_states`.



In [ ]:
decoder_inputs = Input(shape=(None, num_decoder_tokens))

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs, initial_state=encoder_states
)

decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

**Reasoning**:
Now that both the encoder and decoder models have been redefined with the new `latent_dim`, the next step is to re-create the full sequence-to-sequence model and compile it. This prepares the model for retraining with the increased epochs and latent dimension.



In [ ]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)

In [ ]:
for seq_index in range(10):

    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)

    print("Input:", input_data[seq_index])
    print("Predicted:", decoded_sentence.strip())
    print("-" * 40)
